# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print("Dataset loaded successfully!\n")
print(f"Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all record sets, then for each record set, all its fields and the columns, referencing their `@id`.

In [ ]:
# List all available record sets
print("\nAvailable record sets and their fields:\n")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"RecordSet name: {record_set.name}, @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    Field name: {field.name}, @id: {field.id}, data_type: {field.data_type if hasattr(field,'data_type') else 'N/A'}")
            if hasattr(field, 'columns') and field.columns:
                for col in field.columns:
                    # Some columns may be referenced by their id
                    if hasattr(col, 'id'):
                        print(f"      Column @id: {col.id}")
    print("")

# If no record sets found, print warning
if not record_set_ids:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entity references are by their `@id`.

The main record set contains the survey data. We'll extract all available record sets.

In [ ]:
# Extract data from each record set to a pandas DataFrame, referencing by `@id`
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records.")
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}\n")
    else:
        print(f"  No records found for {record_set_id}.")

# For demonstration, pick the first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None

# Show top rows of main DataFrame if available
if main_record_set_id and main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())
else:
    print("No data found to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a GAD-7 numeric field (e.g., `gad7_total_score`) and group by a demographic field (e.g., `gender`). You must substitute these names with the appropriate column `@id` as shown above.

In [ ]:
import numpy as np

# --- You may need to adjust these field IDs to those actually shown above! ---
# For this EDA, we select likely field names/IDs by inspecting the fields list above:
# E.g.,
# gad7_total_score_field_id = '@gad7_total_score'
# gender_field_id = '@gender'
# 
# You may need to inspect the output above or your DataFrame columns.

gad7_score_col_candidates = [col for col in dataframes[main_record_set_id].columns if ('gad7' in col.lower() and 'score' in col.lower())]
gender_col_candidates = [col for col in dataframes[main_record_set_id].columns if 'gender' in col.lower()]

print("Possible GAD-7 score column(s):", gad7_score_col_candidates)
print("Possible gender column(s):", gender_col_candidates)

# Try using the first match for each
if gad7_score_col_candidates:
    gad7_total_score = gad7_score_col_candidates[0]
else:
    gad7_total_score = None

if gender_col_candidates:
    gender_field = gender_col_candidates[0]
else:
    gender_field = None

# Now filter and normalize
if gad7_total_score and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id].copy()
    # Clean numeric
    df[gad7_total_score] = pd.to_numeric(df[gad7_total_score], errors='coerce')
    threshold = 10
    filtered_df = df[df[gad7_total_score] > threshold].copy()
    print(f"Filtered records with {gad7_total_score} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{gad7_total_score}_normalized"] = (
        (filtered_df[gad7_total_score] - filtered_df[gad7_total_score].mean()) /
        filtered_df[gad7_total_score].std()
    )
    print(f"Normalized {gad7_total_score} for filtered records:")
    display(filtered_df[[gad7_total_score, f"{gad7_total_score}_normalized"]].head())

    # Group by gender (if exists)
    if gender_field and gender_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_field)[gad7_total_score].mean().reset_index()
        print(f"\nGrouped mean {gad7_total_score} by {gender_field}:")
        display(grouped_df)
else:
    print("GAD-7 score or main record set not found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
if gad7_total_score and gad7_total_score in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[gad7_total_score].dropna(), kde=True, bins=15, color='skyblue')
    plt.title('GAD-7 Total Score Distribution')
    plt.xlabel('GAD-7 Total Score')
    plt.ylabel('Count')
    plt.show()

# Boxplot of GAD-7 scores by gender
if gad7_total_score and gender_field and gender_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=gender_field, y=gad7_total_score, data=df, showmeans=True)
    plt.title('GAD-7 Scores by Gender')
    plt.xlabel('Gender')
    plt.ylabel('GAD-7 Total Score')
    plt.show()

## 6. Conclusion
We demonstrated how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. We reviewed its schema, extracted tabular data by referencing all entities via their `@id`, performed exploratory analysis, and visualized mental health measures such as GAD-7 scores.

This workflow can be extended to additional fields (e.g., PHQ-9 or PSQ scores) and other record sets. The use of Croissant schemas and `mlcroissant` enables consistent and robust FAIR data practices for AI-ready analyses.